<div style="background: linear-gradient(90deg,#1e3c72,#2a5298); padding:20px; border-radius:10px;">
<h1 style="colo​r:white;">🌍 GDAL GeoTIFF Notebook</h1>
<p style="color:white;">MTG Full Disk Processing</p>
</div>

In [ ]:
# Variables
INPUT="/toto/tata/titi/mon_image.tif"
OUTDIR="RESULTS"
!mkdir -p $OUTDIR

## 🔎 Inspection

In [ ]:
!gdalinfo $INPUT

## 🖼️ Miniature

In [ ]:
THUMB="$OUTDIR/thumb.jpg"
!gdal_translate -of JPEG -outsize 10% 10% $INPUT $THUMB
!display $THUMB

## 🌐 Reprojection

In [ ]:
!gdalwarp -t_srs EPSG:4326 $INPUT $OUTDIR/wgs84.tif
!gdalwarp -t_srs EPSG:3857 $INPUT $OUTDIR/mercator.tif
!gdalwarp -t_srs "+proj=stere +lat_0=90" $INPUT $OUTDIR/stereo.tif
!gdalwarp -t_srs "+proj=ortho +lat_0=0 +lon_0=0" $INPUT $OUTDIR/ortho.tif
!gdalwarp -t_srs "+proj=geos +h=35785831" $INPUT $OUTDIR/geos.tif

## ✂️ Découpage France

In [ ]:
!gdalwarp -t_srs EPSG:4326 -te -5 41 10 52 $INPUT $OUTDIR/france.tif

## 🎨 Amélioration

In [ ]:
!gdal_translate -exponent 0.8 $INPUT $OUTDIR/gamma.tif
!gdal_translate -scale 0 255 0 255 $INPUT $OUTDIR/stretch.tif

## 🗺️ Trait de côte

In [ ]:
!gdal_rasterize -burn 255 -burn 255 -burn 255 -l coastlines coastlines.shp $OUTDIR/france.tif

## 🔤 Texte

In [ ]:
!convert $OUTDIR/france.tif -gravity South -pointsize 40 -fill white -annotate +0+20 "France - MTG" $OUTDIR/france_text.tif

## 📍 Extraction pixel

In [ ]:
!gdallocationinfo -wgs84 $INPUT 1.5 46.5
!gdallocationinfo -valonly -wgs84 $INPUT 1.5 46.5

## 🎭 Palette

In [ ]:
!gdal_translate -b 1 $INPUT $OUTDIR/gray.tif
echo "0 0 0 255
128 0 255 0
255 255 0 0" > palette.txt
!gdaldem color-relief $OUTDIR/gray.tif palette.txt $OUTDIR/color.tif

## 📐 Resize

In [ ]:
!convert $INPUT -resize 1920x1080 $OUTDIR/resized.jpg

## ⚡ Pipeline

In [ ]:
!gdalwarp -t_srs EPSG:4326 $INPUT $OUTDIR/tmp1.tif
!gdal_translate -exponent 0.9 $OUTDIR/tmp1.tif $OUTDIR/tmp2.tif
!gdalwarp -te -5 41 10 52 $OUTDIR/tmp2.tif $OUTDIR/final.tif
!gdal_translate -of JPEG -outsize 20% 20% $OUTDIR/final.tif $OUTDIR/thumb_final.jpg
!display $OUTDIR/thumb_final.jpg

## 📊 Bonus

In [ ]:
!gdalinfo -hist $INPUT
!gdalwarp -cutline france.shp -crop_to_cutline $INPUT $OUTDIR/france_mask.tif
!gdal_translate -co COMPRESS=DEFLATE -co PREDICTOR=2 $INPUT $OUTDIR/compressed.tif
!gdal_translate -of COG $INPUT $OUTDIR/cog.tif